# 11 · Additive models (GAM, EBM) and other model ideas

Tests the ideas from the "what else could improve it" list. The winner was the simplest: a **GAM built from boosted stumps** (every tree asks one single question, so each feature gets its own curve and the curves are added).

**Needs:** `features3.parquet` (03), `search.json` (04). Optional: `features4.parquet`. **Writes:** `v3_<name>_oof.npy` / `_test.npy`. **Time:** the default run below takes about 45 minutes; `spline` and `gamtune` are optional extras.

| Part | Out-of-fold AUC | Verdict |
|---|---|---|
| `gam` (boosted stumps) | 0.7524 | best single model |
| `gamtune` (5 variants) | 0.7522 - 0.7524 | same |
| `ebm` 100 features | 0.7512 | good, different |
| `ebm` 150 features + 20 interactions | 0.7505 | interactions don't help |
| `ebm` 250 features | 0.7498 | too many features hurts |
| `lintree` (linear leaves) | 0.7461 | adds a little diversity |
| `mono` (monotone constraints) | 0.7507 | no gain |
| `rank` (pairwise ranking objective) | 0.7459 | no gain |
| `lrfeat` (cross-fitted logistic score as a feature) | 0.7508 | no gain |
| `dart` | 0.668 | failed |

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy interpret

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, xgboost as xgb, json, sys, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer

def main(part, EBM_NF=100, EBM_INTER=0):
    W=WORK; D=DATA
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    X=pd.read_parquet(f'{W}/features3.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    catc=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']; Xg=X.copy()
    for c in catc: Xg[c]=Xg[c].astype('category')
    T=Xg.merge(tr,on='encounter_id'); S=te.merge(Xg,on='encounter_id',how='left'); y=T.readmit_30d.values; assert S.encounter_id.tolist()==te.encounter_id.tolist()
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); t0=time.time()
    best=json.load(open(f'{W}/search.json'))[0][1]; base=dict(objective='binary',learning_rate=0.05,bagging_freq=1,verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
    def save(name,o,t): np.save(f'{W}/v3_{name}_oof.npy',o); np.save(f'{W}/v3_{name}_test.npy',t); print(name,'OOF AUC %.5f'%roc_auc_score(y,o),'t=%.0f'%(time.time()-t0),flush=True)
    def run_lgb(p,fs,rounds=5000,es=True,extra=None):
        o=np.zeros(len(T)); t=np.zeros(len(S))
        for f,(a,b) in enumerate(folds):
            Ta,Tb,Ts=T.loc[a,fs],T.loc[b,fs],S[fs]
            if extra is not None: Ta,Tb,Ts=extra(f,a,b,Ta,Tb,Ts)
            if es: m=lgb.train({**p,'seed':f},lgb.Dataset(Ta,y[a]),rounds,valid_sets=[lgb.Dataset(Tb,y[b])],callbacks=[lgb.early_stopping(200,verbose=False)]); it=m.best_iteration
            else: m=lgb.train({**p,'seed':f},lgb.Dataset(Ta,y[a]),rounds); it=None
            o[b]=m.predict(Tb,num_iteration=it); t+=m.predict(Ts,num_iteration=it)/5
        return o,t
    if part=='gam':   # boosted stumps = a true additive model (one curve per feature)
        p={**base,'num_leaves':2,'max_depth':1,'learning_rate':0.1,'min_child_samples':100,'feature_fraction':0.5,'lambda_l2':20,'max_bin':63}
        save('gam_stumps',*run_lgb(p,feats))
    if part=='mono':  # monotone constraints on features with known one-directional effect
        up=('lab_creatinine_','lab_bun_','lab_bnp_','sev_','kidney_sev','n_critical_last'); dn=('lab_albumin_','lab_hemoglobin_','egfr_')
        exact_up={'age','pu_total','pu_acute_365','pu_acute_180','pu_ED_365','pu_IP_365','n_chronic','n_dx','charlson_like','lace','deprivation_index','los_days','dp_followup_missing'}
        mc=[1 if (f.startswith(up) and not f.endswith(('_h_last','_n','_hfrac','_lfrac','_slope','_std','_peak_h_to_dc'))) or f in exact_up else (-1 if f.startswith(dn) and not f.endswith(('_h_last','_n','_hfrac','_lfrac','_slope','_std','_peak_h_to_dc','_delta')) else 0) for f in feats]
        print('constrained features:',sum(m!=0 for m in mc),flush=True)
        p={**base,**best,'monotone_constraints':mc,'monotone_constraints_method':'advanced'}; save('mono',*run_lgb(p,feats))
    if part=='lintree':  # LightGBM with linear models inside the leaves (numeric features only)
        nf=[f for f in feats if f not in catc]; p={**base,**best,'linear_tree':True,'linear_lambda':10.0,'learning_rate':0.04}
        save('lintree',*run_lgb(p,nf))
    if part=='dart':
        p={**base,**best,'boosting':'dart','drop_rate':0.1,'skip_drop':0.5,'learning_rate':0.1}; save('dart',*run_lgb(p,feats,rounds=450,es=False))
    if part=='rank':  # train the model to rank readmitted patients above others (pairwise)
        P=dict(objective='rank:pairwise',eval_metric='auc',tree_method='hist',max_depth=3,eta=0.05,min_child_weight=20,subsample=0.8,colsample_bytree=0.3,reg_lambda=20,max_bin=63,max_cat_to_onehot=8,nthread=2,lambdarank_pair_method='mean',lambdarank_num_pair_per_sample=8)
        dS=xgb.DMatrix(S[feats],enable_categorical=True); o=np.zeros(len(T)); t=np.zeros(len(S))
        for f,(a,b) in enumerate(folds):
            perm=np.random.RandomState(f).permutation(a); qid=np.arange(len(perm))//3000
            da=xgb.DMatrix(T.loc[perm,feats],y[perm],qid=qid,enable_categorical=True); db=xgb.DMatrix(T.loc[b,feats],y[b],qid=np.zeros(len(b),dtype=int),enable_categorical=True)
            m=xgb.train({**P,'seed':f},da,3000,evals=[(db,'v')],early_stopping_rounds=200,verbose_eval=False); n=m.best_iteration+1
            o[b]=m.predict(db,iteration_range=(0,n)); t+=m.predict(dS,iteration_range=(0,n))/5; print(' rank fold',f,n,'%.5f'%roc_auc_score(y[b],o[b]),flush=True)
        save('rank',o,t)
    def build_Z():
        num=[c for c in feats if c not in catc and T[c].notna().sum()>0 and T[c].nunique()>1]; cat=[c for c in feats if c in catc]
        A=pd.concat([T[num],S[num]]); miss=A.columns[A.isna().mean()>0.01]; lo,hi=T[num].quantile(.005),T[num].quantile(.995); med=T[num].clip(lo,hi,axis=1).median()
        N=lambda df: df[num].clip(lo,hi,axis=1).fillna(med).values.astype(np.float32)
        R=lambda df: pd.concat([df[miss].isna().astype(np.float32).add_suffix('_na'),df[cat].astype(object).fillna('NA')],axis=1)
        Rall=pd.get_dummies(pd.concat([R(T),R(S)],ignore_index=True),columns=cat,dtype=np.float32)
        return N(T),N(S),Rall.iloc[:len(T)].values,Rall.iloc[len(T):].values,num
    if part=='spline':  # logistic regression where every important factor gets a flexible curve
        Nt,Ns,Rt,Rs,num=build_Z(); res={}
        for Cc in [0.01,0.03]:
            o=np.zeros(len(T)); t=np.zeros(len(S))
            for f,(a,b) in enumerate(folds):
                au=np.array([abs(roc_auc_score(y[a],Nt[a][:,j])-.5) for j in range(Nt.shape[1])]); sel=np.argsort(-au)[:120]; rest=np.setdiff1d(np.arange(Nt.shape[1]),sel)
                st=SplineTransformer(n_knots=6,degree=3,knots='quantile',extrapolation='constant').fit(Nt[a][:,sel])
                mk=lambda N,R: np.hstack([st.transform(N[:,sel]),N[:,rest],R])
                Xa,Xb,Xs=mk(Nt[a],Rt[a]),mk(Nt[b],Rt[b]),mk(Ns,Rs); sc=StandardScaler().fit(Xa); Xa,Xb,Xs=[np.clip(sc.transform(M),-6,6) for M in (Xa,Xb,Xs)]
                m=LogisticRegression(C=Cc,max_iter=400).fit(Xa,y[a]); o[b]=m.decision_function(Xb); t+=m.decision_function(Xs)/5
                print(' spline C',Cc,'fold',f,'%.5f'%roc_auc_score(y[b],o[b]),'t=%.0f'%(time.time()-t0),flush=True)
            res[Cc]=(roc_auc_score(y,o),o,t)
        kb=max(res,key=lambda k:res[k][0]); print('best C',kb); o,t=res[kb][1:]; save('spline_lr',1/(1+np.exp(-o)),1/(1+np.exp(-t)))
    if part=='lrfeat':  # cross-fitted linear score handed to LightGBM as an extra feature
        Nt,Ns,Rt,Rs,num=build_Z(); Zt=np.hstack([Nt,Rt]); Zs=np.hstack([Ns,Rs])
        def lr_fit(a_idx): sc=StandardScaler().fit(Zt[a_idx]); m=LogisticRegression(C=0.02,max_iter=300).fit(np.clip(sc.transform(Zt[a_idx]),-6,6),y[a_idx]); return lambda M: m.decision_function(np.clip(sc.transform(M),-6,6))
        def extra(f,a,b,Ta,Tb,Ts):
            inner=np.zeros(len(a))
            for ia,ib in StratifiedKFold(5,shuffle=True,random_state=100+f).split(a,y[a]): inner[ib]=lr_fit(a[ia])(Zt[a[ib]])
            g=lr_fit(a); Ta,Tb,Ts=Ta.copy(),Tb.copy(),Ts.copy(); Ta['lr_score']=inner; Tb['lr_score']=g(Zt[b]); Ts['lr_score']=g(Zs); print(' lrfeat fold',f,'LR alone %.5f'%roc_auc_score(y[b],Tb.lr_score),'t=%.0f'%(time.time()-t0),flush=True); return Ta,Tb,Ts
        save('lrfeat_lgb',*run_lgb({**base,**best},feats,extra=extra))
    if part=='ebm':  # Explainable Boosting Machine: additive curves (+ a few pairwise interactions) on the 100 strongest features
        from interpret.glassbox import ExplainableBoostingClassifier as EBC
        o=np.zeros(len(T)); t=np.zeros(len(S)); nf=EBM_NF; inter=EBM_INTER
        p={**base,**best,'learning_rate':0.06}
        for f,(a,b) in enumerate(folds):
            m0=lgb.train({**p,'seed':f},lgb.Dataset(T.loc[a,feats],y[a]),300); g=pd.Series(m0.feature_importance('gain'),index=feats).sort_values(ascending=False); fs=list(g.index[:nf])
            cv=lambda d: d[fs].copy()
            Ta,Tb,Ts=cv(T.loc[a]),cv(T.loc[b]),cv(S)
            for c in fs:
                if c in catc: 
                    for d in (Ta,Tb,Ts): d[c]=d[c].astype(object).where(d[c].notna(),'NA').astype(str)
            m=EBC(interactions=inter,outer_bags=4,inner_bags=0,learning_rate=0.02,max_rounds=3000,early_stopping_rounds=100,min_samples_leaf=50,random_state=f,n_jobs=2).fit(Ta,y[a])
            o[b]=m.predict_proba(Tb)[:,1]; t+=m.predict_proba(Ts)[:,1]/5; print(' ebm fold',f,'%.5f'%roc_auc_score(y[b],o[b]),'t=%.0f'%(time.time()-t0),flush=True)
        save(f'ebm_{nf}_{inter}',o,t)
    if part=='gamtune':
        G={'A':dict(feature_fraction=0.3,min_child_samples=100,lambda_l2=20,max_bin=63,learning_rate=0.05),
           'B':dict(feature_fraction=0.8,min_child_samples=100,lambda_l2=20,max_bin=63,learning_rate=0.05),
           'C':dict(feature_fraction=0.5,min_child_samples=300,lambda_l2=100,max_bin=63,learning_rate=0.05),
           'D':dict(feature_fraction=0.5,min_child_samples=100,lambda_l2=20,max_bin=63,learning_rate=0.05,num_leaves=3,max_depth=2),
           'E':dict(feature_fraction=0.5,min_child_samples=100,lambda_l2=20,max_bin=255,learning_rate=0.05)}
        for k,q in G.items():
            p={**base,'num_leaves':2,'max_depth':1,**q}; save(f'gam_{k}',*run_lgb(p,feats))

In [ ]:
main('gam')                                   # boosted-stump GAM (the best single model)
main('mono'); main('lintree'); main('rank'); main('lrfeat')
main('ebm', 100, 0); main('ebm', 150, 20); main('ebm', 250, 0)
# optional / slower:  main('gamtune')  main('spline')  main('dart')